## Load segments from Slicer

In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, tetra_mesh_from_stl, tetra_shell_from_two_surfaces, set_the_offset


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_0.5_fat_AIMax/pixel_spacing.txt")

base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/"
in_folder = ""    

folder = "Meshes/"

if not os.path.isdir(base + folder):
    os.mkdir(base + folder)




filename = "segmentations_pdt1_v160725.seg.nrrd"
export_name = base + folder + filename[:-8]


data, header = nrrd.read(base + in_folder + filename)



id_max = numpy.max(data)

print("pixel spacing = ",pixel_spacing)
print("id = ", id_max )

print("dimensions = ", data.shape)


factor = pixel_spacing[2]/pixel_spacing[1]
data = ndimage.zoom(data, [1,1,factor], order = 0)

data = ndimage.median_filter(data, size=5)



print("dimensions = ", data.shape)

id = int(numpy.max(data))
print("id = ", id)


pixel spacing =  [0.3125 0.3125 0.5   ]
id =  7
dimensions =  (145, 209, 101)
dimensions =  (145, 209, 162)
id =  7


In [ ]:
for id in range(1,numpy.max(data)+1):
# for id in range(3,4):
    print("id = ", id)


    if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

        mask = numpy.zeros(data.shape, dtype=numpy.uint8)
        mask[(data>id-0.5) & (data<id+0.5 )]=1

        print(len(numpy.where(mask==1)[0]))

        mask = ndimage.median_filter(mask, size=7)
        mask[mask<0.5]=0
        mask[mask>=0.5]=1

        numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

        if id ==3:
            mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], True )

            mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
            # compute mean voxel position
            offset = mask_voxels.mean(axis=0)*pixel_spacing[0]  # [z_mean, y_mean, x_mean]
            numpy.savetxt(base + folder + "offset.txt", offset)

        else:
            mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )



id =  1
24463
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_1.stl
id =  2
id =  3
261394
Problem edges: 0
Outer and inner surfaces written.
id =  4
22123
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_4.stl
id =  5
20078
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_5.stl
id =  6
17110
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_6.stl
id =  7
18597
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_7.stl


In [3]:
h=1.0

out_folder = "re-meshed_h_"+str(h)+"/"

if not os.path.isdir(base + folder + out_folder):
    os.mkdir(base + folder + out_folder)


for id in range(1,numpy.max(data)+1):

    if id !=3:
        print("id = ", id) 
        

        stl_file = export_name +"_surf_id_"+str(id)+".stl"

        if os.path.isfile(stl_file):
            tetra_mesh_from_stl(stl_file, base + folder + out_folder + "id_"+str(id)+"_" + "_h_"+str(h), h)
            set_the_offset(base + folder + out_folder + "id_"+str(id)+"_" + "_h_"+str(h), offset)


id =  1
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_1.stl'...
Info    : 17394 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_1.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 17394 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 8818 triangles split in 2 parts because parametrized triangles are too small (4.08645e-09)
Info    :  - Level 2 partition with 4324 triangles split in 2 parts because parametrized triangles are too small (9.89302e-09)
Info    :  - Level 3 partition with 2120 triangles split in 2 parts because parametrized triangles are too small (2.02646e-09)
Info    : Found 5 model su

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

id =  2
id =  4
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_4.stl'...
Info    : 18824 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_4.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 18824 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.171517s, CPU 0.16891s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 8.00006e-06s, CPU 7e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.0705032s, CPU 0.0685

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

id =  5
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_5.stl'...
Info    : 14104 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_5.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 14104 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.125969s, CPU 0.124026s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 6.33486e-06s, CPU 6e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.0509191s, CPU 0.050181s)   

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

id =  6
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_6.stl'...
Info    : 13740 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_6.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 13740 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 6818 triangles split in 2 parts because parametrized triangles are too small (6.45782e-09)
Info    :  - Level 2 partition with 3423 triangles split in 2 parts because parametrized triangles are too small (2.02939e-09)
Info    : Found 4 model surfaces
Info    : Found 3 model curves
Info    : Done classifying surfaces (Wall 0.171464s, CPU 0.168315s)
Info    : Creating ge

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

id =  7
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_7.stl'...
Info    : 13576 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_7.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 13576 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.121348s, CPU 0.119913s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 6.45965e-06s, CPU 6e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.0496244s, CPU 0.048545s)   

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

In [4]:
id = 3
stl_file = export_name +"_surf_id_"+str(id)+".stl"


tetra_shell_from_two_surfaces(stl_file.replace(".stl", "_outer.stl"), stl_file.replace(".stl", "_inner.stl"), base + folder + out_folder + "id_"+str(id)+"_shell_" + "_h_"+str(h), h)
set_the_offset(base + folder + out_folder + "id_"+str(id)+"_shell_" + "_h_"+str(h), offset)


Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_3_outer.stl'...
Info    : 60128 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_3_outer.stl'
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_3_inner.stl'...
Info    : 60128 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/segmentations_pdt1_v160725._surf_id_3_inner.stl'
Info    : Classifying surfaces (angle: 40)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 60128 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 0 partiti

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.